---

# 10. Generalized Parsing
**[Emil Sekerinski](http://www.cas.mcmaster.ca/~emil/), McMaster University, March 2026**

---

### Functional Parsing

Functional parsing is a style of writing recursive-descent parsers that can lead to particularly compact implementations in languages that allow *matching* a prefix of a sequence. If `T` is the set of terminals, then a parser is a function of type `seq T → seq T` that consumes a prefix of its input and returns the remainder; there are no global variables.

Functional languages like OCaml and Haskell allow combining function definitions with matching. If there is no match, an exception is raised. Consider the EBNF grammar with the sole production `A → 'a' A 'c' | 'b'`. A parser for `A` in Haskell is:

```haskell
a :: String -> String
a ('a':t) =
  case a t of
    ('c':u) -> u
a ('b':t) = t
```

Like in recursive descent parsing, there is one parsing function for each nonterminal. OCaml and Haskell require functions to start with a lowercase letter. An input is parsed if the parsing function of the start symbol returns the empty sequence:

```haskell
ghci> a "abc"
""
ghci> a "abcd"
"d"
ghci> a "ac"
"***Exception: Non-exhaustive patterns in function a
```

To reproduce these messages, `ghci` can be started from the command line in a separate window. That starts an interactive session, to which the code above can be pasted. Multi-line definitions must be wrapped in `:{` and `:}`, e.g.:

```haskell
ghci> :{
ghci| a :: String -> String
ghci| a ('a':t) =
ghci|   case a t of
ghci|     ('c':u) -> u
ghci| a ('b':t) = t
ghci| :}
ghci> s "abc"
""
```

In general, for each production `p` of an EBNF grammar, a parsing function `Pr p` is defined. For production `B → E`, the name of the function is `B`, and its body is `Pr E`, the function body recognizing `E`:

| `p`             | `Pr p`                       |
|:----------------|:-----------------------------|
| `B → E`         | `let rec B = Pr E ;;`        |

The rules for constructing `Pr E` are:

| `E`             | `Pr E `                               |
|:----------------|:--------------------------------------|
| `'a'`           | `function 'a' :: t -> t`              |
| `B`             | `B`                                   |
| `(E₁)`          | `Pr E₁`                               |
| `[E₁]`          | `function s ->`<br>`    match s with first(E₁)::_ -> Pr E₁ \| _ -> s`|
| `{E₁}`          | `let rec h = function s ->`<br>`    match s with first(E₁)::_ -> h(Pr E₁ s) \| _ -> s`<br>`in h`|
| `E₁ E₂ …`       | `function s -> … (Pr E₂ (Pr E₁ s))` |
| `E₁ │ E₂ │ …`   | `function s -> match s with`<br>`    first(E₁)::_ -> Pr E₁ s \|`<br>`    first(E₂)::_ -> Pr E₂ s \|`<br>`    …` |

For `{E₁}`, an auxiliary recursive function, `h` is introduced. The procedure of the start symbol has to be called to recognize a sentence of the language.

For example, the parser for `'a' A 'c'` is constructed as follows:

<div style="float:left">

```ocaml
     Pr ('a' A 'c')
=  function s -> Pr 'c' (Pr A (Pr 'a' s))
=  function s -> Pr 'c' (Pr A ((function 'a' :: t -> t) s))
=  function s -> Pr 'c' (A ((function 'a' :: t -> t) s))
=  function s -> Pr 'c' ((function 'a' :: t -> A t) s)
=  function s -> (function 'c' :: t -> t) (A ((function 'a' :: t -> t) s))
```

</div>
<div style="float:left">

```
    definition of Pr ('a' A 'c')
    definition of Pr 'a' 
    definition of Pr A 
    simplification
    definition of Pr 'c'
    
```

</div>

The parser for `S → 'a' A 'c' | 'b'` is then:

```ocaml
     Pr(A → 'a' A 'c' | 'b')
=  let rec A = Pr ('a' A 'c' | 'b') ;;
=  let rec A = function s -> match s with
       'a'::_ -> Pr ('a' A 'c') s |
       'b'::_ -> Pr 'b' s ;;
=  let rec A = function s -> match s with
       'a'::_ -> (function 'c' :: t -> t) (A ((function 'a' :: t -> t) s)) |
       'b'::_ -> (function 'b' :: t -> t) s ;;
=  let rec A = function s -> match s with
       'a'::t -> (function 'c' :: t -> t) (A t) |
       'b'::t -> t ;;
=  let rec A = function
       'a'::t -> (function 'c' :: t -> t) (A t) |
       'b'::t -> t ;;
=  let rec A = function
       'a'::t -> match A t with 'c' :: t -> t |
       'b'::t -> t ;;
```

As OCaml functions cannot start with a capital letter, `A` is renamed to `aa`.

Python has a `match` statement that matches sequences, like lists, but not strings, against patterns. It cannot be combined with function definitions. If there is no match, execution continues:

In [ ]:
for s in (['a', 'b', 'c'], ['a', 'b', 'c', 'd'], ['b', 'c', 'd', 'e'], ['b'], 'abc'):
    match s:
        case ['a', second, third]: print('second =', second, ' third =', third) # matches exactly 3
        case ['b', *tail]: print('tail = ', tail) # matches 1 or more

Below is the corresponding Python implementation; `case ['a', *t]` is successful if `s` starts with `a` and assigns the list of remaining elements to `t`. The subsequent call to `A(t)` returns the remainder after parsing `A`, which, according to the grammar, is supposed to be `c`.

In [ ]:
def A(s: list[str]) -> list[str]: # A → 'a' A 'c' | 'b'
    #print(s)
    match s:
        case ['a', *t]:
            match A(t):
                case ['c', *u]: return u
                case _: raise Exception("'c' expected")
        case ['b', *t]: return t
        case _: raise Exception("'a' or 'b' expected")

assert A(['a', 'a', 'b', 'c', 'c']) == []

Python distinguishes statements from expressions. An anonymous function (`lambda`) is an expression, while `match` is a statement; hence, these cannot be combined. Instead, each parsing function declares a parameter for the input, `s` below, which is passed to `Pr E s` for constructing a statement that takes `s` as input and recognizes `E`:

| `p`             | `Pr p`                       |
|:----------------|:-----------------------------|
| `B → E`         | `def B(s) = Pr E s`          |

The rules for constructing `Pr E s` are:

| `E`             | `Pr E s`                            |
|:----------------|:------------------------------------|
| `'a'`           | `match s:`<br>`    case ['a', *t]: return t`<br>`    case _: raise` |
| `B`             | `return B(s)`                       |
| `(E₁)`          | `Pr E₁ s`                           |
| `[E₁]`          | `match s:`<br>`    case [first(E₁), *t]: Pr E₁ s`<br>`    case _: return s` |
| `{E₁}`          | `def h(s):`<br>`    match s:`<br>`        case [first(E₁),_]: h(Pr E₁ s)`<br>`        case _: return s`<br>`return h(s)` |
| `E₁ E₂ …`       | `… (Pr E₂ (Pr E₁ s))`               |
| `E₁ │ E₂ │ …`   | `match s:`<br>`    case [first(E₁), *t]: Pr E₁ s`<br>`    case [first(E₂), *t]: Pr E₂ s`<br>`    …` |

For `{E₁}`, an auxiliary recursive function, `h` is introduced. The procedure of the start symbol must be called to recognize a sentence in the language. The above parser for `A` is derived using this scheme and simplifications:

<div style="float:left">

```python
     Pr ('a' A 'c') s
=  Pr 'c' (Pr A (Pr 'a' s))
=  Pr 'c' (Pr A (match s:
    case ['a', *t]: return t
    case _: raise))
=  Pr 'c' (return A (match s:
    case ['a', *t]: return t
    case _: raise))
=  Pr 'c' ((match s:
    case ['a', *t]: return A(t)
    case _: raise))
=  (match (match s:
    case ['a', *t]: return A(t)
    case _: raise):
    case ['c', *t]: return t
        case _: raise
=  (match s:
    case ['a', *t]:
        match A(t)
            case ['c', *t]: return t
            case _: raise
    case _: raise)
```

</div>
<div style="float:left">

```
    definition of Pr ('a' A 'c') s
    definition of Pr 'a' s
    definition of Pr A 

    
    simplification  

    
    definition of Pr 'c'

    
    simplification  



    
```

</div>


We continue with OCaml and Haskell due to brevity. The goal of parsing is not just to accept the input, but to construct an abstract syntax tree. For the EBNF grammar

    expression → term { ('+' | '-') term }
    term → factor { ( '*' | '/') factor }
    factor → id | id '(' exprList ')' | '(' expression ')
    exprList → expression { ',' expression }
    id → 'a' | … | 'z'
    
the type of the abstract syntax tree is:

```haskell
data Exp
  = Sum Exp Exp
  | Diff Exp Exp
  | Prod Exp Exp
  | Quot Exp Exp
  | Id Char
  | Fun Char [Exp]
```

For example, the abstract syntax tree of `a+f(x,y)` is:

    (Sum (Id 'a') (Fun 'f' [Id 'x',Id 'y'])


A parser is a function of type `String -> (Exp, String)`: it consumes a prefix of its input and returns the abstract syntax tree of the consumed part and the remainder of the input. The rules for constructing `Pr E` are modified depending on how the abstract syntax tree is to be constructed:

| `E`           | `Pr E `                               | Condition |
|:--------------|:--------------------------------------|-----------|
| `E F`         | `function s ->`<br>`    let (p, t) = Pr E s`<br>`    in let (q, u) = Pr F t`<br>`    in (Cons(p, q), u)` | parse trees of `E`, `F` of types `P`, `Q`<br>resulting parse tree of type `R`<br>result to be constructed with `Cons: P * Q -> R`
| `E {sep E}`   | `function s ->`<br>`    let rec h = function`<br>`    \| (p, sep::t) -> let (q, u) = h (Pr E t) in (p::q, u)`<br>`    \| (p, t) -> ([p], t)`<br>`    in h(Pr E s)` | parse tree of `E` of type `P`<br>tree for `E sep ... sep E` to be constructed as a flat list<br>auxiliary function `h` of type `P * T list -> P list * T list`
| `E {lop E}`   | `function s ->`<br>`    let rec h = function`<br>`    \| (p, lop::t) -> let (q, u) = Pr E t in h(Op(p, q), u)`<br>`    \| (p, t) -> (p, t)`<br>`    in h(Pr E s)` | parse tree of `E` of type `P`<br>tree for `E lop ... lop E` to be constructed with `Op: P * P -> P` left associative<br>auxiliary function `h` of type `P * T list -> P * T list`

The parser for the above expression grammar becomes:
```haskell
expression :: String -> (Exp, String)
expression s = moreterms (term s)
  where
    moreterms :: (Exp, String) -> (Exp, String)
    moreterms (p, '+':t) =
      let (q, u) = term t in moreterms (Sum p q, u)
    moreterms (p, '-':t) =
      let (q, u) = term t in moreterms (Diff p q, u)
    moreterms any = any

term :: String -> (Exp, String)
term s = morefactors (factor s)
  where
    morefactors :: (Exp, String) -> (Exp, String)
    morefactors (p, '*':t) =
      let (q, u) = factor t in morefactors (Prod p q, u)
    morefactors (p, '/':t) =
      let (q, u) = factor t in morefactors (Quot p q, u)
    morefactors any = any

factor :: String -> (Exp, String)
factor ('(':t) =
  case expression t of
    (p, ')':u) -> (p, u)
    _          -> error "missing )"
factor (id:'(':t) | id >= 'a' && id <= 'z' =
  case exprList t of
    (ps, ')':u) -> (Fun id ps, u)
    _           -> error "missing )"
factor (id:t) | id >= 'a' && id <= 'z' = (Id id, t)
factor (_:_) = error "id or ( expected"
factor []    = error "unexpected end"

exprList :: String -> ([Exp], String)
exprList s =
  case expression s of
    (p, ',':u) -> let (q, n) = exprList u in (p:q, n)
    (p, t)     -> ([p], t)
```

<!-- ```ocaml
let rec expression s =
  let rec moreterms = function
      | (p, '+'::t) -> let (q, u) = term t in moreterms (Sum (p, q), u)
      | (p, '-'::t) -> let (q, u) = term t in moreterms (Diff (p, q), u)
      | any -> any
  in moreterms (term s)

and term s =
  let rec morefactors = function
    | (p, '*'::t) -> let (q, u) = factor t in morefactors (Prod (p, q), u)
    | (p, '/'::t) -> let (q, u) = factor t in morefactors (Quot (p, q), u)
    | any -> any
  in morefactors (factor s)

and factor = function
  | '('::t ->
      (match expression t with
        | (p, ')'::u) -> (p, u)
        | _ -> raise (Failure "missing )"))
  | id::'('::t when id >= 'a' && id <= 'z' ->
      (match exprList t with
        | (p, ')'::u) -> (Fun(id, p), u)
        | _ -> raise (Failure "missing )"))
  | id::t when id >= 'a' && id <= 'z' -> (Id (id), t)
  | _::_ -> raise (Failure "id or ( expected")
  | [] -> raise (Failure "unexpected end")

and exprList s =
    match expression s with
      | (p, ','::t) -> let (q, n) = exprList t in (p::q, n)
      | (p, t) -> ([p], t)
;;
``` -->

For example:
```haskell
ghci> expression "a*(b+c)"
(Prod (Id 'a') (Sum (Id 'b') (Id 'c')),"")
ghci> expression "a+f(x,y)"
(Sum (Id 'a') (Fun 'f' [Id 'x',Id 'y']),"")
```

Note that the grammar is not `LL(1)` because of:

    factor → id | id '( exprList ') | '(' expression ')
    
Here, a lookahead of two symbols is sufficient to resolve the conflict. The scheme can be generalized to a lookahead of `k` symbols in languages that allow matching the prefix of a sequence, not just the first element. Hence, `LL(k)` grammars can be parsed:

| `E`             | `Pr E s`                            |
|:----------------|:------------------------------------|
| `E₁ │ E₂ │ …`   | `match s:`<br>`    case [firstₖ(E₁), *t]: Pr E₁ s`<br>`    case [firstₖ(E₂), *t]: Pr E₂ s`<br>`    …` |

To produce a meaningful error message, the current position must be maintained. A parser is a function of type

    (Int, String) -> (Int, String)
    
that takes the input together with its starting position and returns a suffix together with its starting position:

```haskell
type Source = (Int, String)

sourceError :: Int -> String -> a
sourceError p m = error ("error at " ++ show p ++ ": " ++ m)

expression :: Source -> Source
expression s =
  case term s of
    (q, '+':t) -> expression (q + 1, t)
    (q, '-':t) -> expression (q + 1, t)
    any        -> any

term :: Source -> Source
term s =
  case factor s of
    (q, '*':t) -> term (q + 1, t)
    (q, '/':t) -> term (q + 1, t)
    any        -> any

factor :: Source -> Source
factor (p, '(':t) =
  case expression (p + 1, t) of
    (q, ')':u) -> (q + 1, u)
    (q, _)     -> sourceError q "missing )"
factor (p, id:'(':t) | id >= 'a' && id <= 'z' =
  case exprList (p + 2, t) of
    (r, ')':u) -> (r + 1, u)
    (r, _)     -> sourceError r "missing )"
factor (p, id:t) | id >= 'a' && id <= 'z' = (p + 1, t)
factor (p, _ : _) = sourceError p "id or ( expected"
factor (p, [])    = sourceError p "unexpected end"

exprList :: Source -> Source
exprList s =
  case expression s of
    (q, ',':t) -> exprList (q + 1, t)
    any        -> any

parse :: String -> IO ()
parse s =
  case expression (1, s) of
    (p, t) -> putStrLn (show (p - 1) ++ " characters parsed")
```
<!-- 
```ocaml
type source = int * char list;;
exception Source_error of int * string ;;

let rec expression (p, s) =
  match term (p, s) with
    | q, '+'::t -> expression (q + 1, t)
    | q, '-'::t -> expression (q + 1, t)
    | any -> any

and term (p, s) =
  match factor (p, s) with
    | q, '*'::t -> term (q + 1, t)
    | q, '/'::t -> term (q + 1, t)
    | any -> any

and factor = function
  | p, '('::t ->
      (match expression (p + 1, t) with
        | q, ')'::u -> (q + 1, u)
        | q, _ -> raise (Source_error (q, "missing )")))
  | p, id::'('::t when id >= 'a' && id <= 'z' ->
      (match exprList (p + 2, t) with
        | r, ')'::u -> (r + 1, u)
        | r, _ -> raise (Source_error (r, "missing )")))
  | p, id::t when id >= 'a' && id <= 'z' -> (p + 1, t)
  | p, _::_ -> raise (Source_error (p, "id or ( expected"))
  | p, [] -> raise (Source_error (p, "unexpected end"))

and exprList (p, s) =
    match expression (p, s) with
      | q, ','::t -> exprList (q + 1, t)
      | any -> any
;;

let parse s =
  try
    match expression (1, s) with
      p, t -> print_string (string_of_int (p - 1) ^ " characters parsed\n")
  with Source_error (p, m) ->
    print_string ("error at " ^ string_of_int p ^ ": " ^ m ^ "\n") ;;
``` -->

For example:

```haskell
ghci> parse "a+(b/c)"
7 characters parsed
ghci> parse "a+(b"
*** Exception: error at 5: missing )
```

For separating scanning from parsing, we can scan the whole input first:

```ocaml
scanner : char list -> symbol list
parser : symbol list -> P * symbol list
```

For the parser to report error positions, we add a position to each symbol:

```ocaml
scanner : char list -> (position * symbol) list
parser : (position * symbol) list -> P * (position * symbol list)
```

The effect of first scanning the whole input and then parsing is that the scanner first reports errors, which may be at the end of the input, and then the parser reports errors, which may be at the beginning of the input. To the programmer, that may appear confusing, as fixing an error at the end may lead to an error being reported at the beginning. To avoid this, scanning can proceed "lazily" as needed.

Consider the following grammar for expressions; it includes two keywords, `div` and `mod`, integer constants, and multi-letter identifiers:

```ebnf
  expression = term { ("+" | "-") term }
  term = factor { ( "*" | "div" | "mod") factor }
  factor = integer | id | id "(" exprList ")" | "(" expression ")"
  exprList = expression { "," expression }
```

where symbols can be separated by blanks and are given by:

```ebnf
  integer = digit {digit}
  id = letter {letter | digit}
  digit = '0' | ... | '9'
  letter = 'a' | ... | 'z'
```

Here is the resulting parser, with scanning, abstract syntax tree generation, and error-position reporting. The scanner, function `symbol`, is of type `source -> symbol * source`, i.e., it scans only the next symbol; confusingly, `symbol` is used both as a function and type. All parser functions are of type `source -> exp * source`.

```ocaml
type exp =
  | Sum of exp * exp
  | Diff of exp * exp
  | Prod of exp * exp
  | Quot of exp * exp
  | Int of int
  | Id of char list
  | Fun of char list * exp list ;;

type symbol = COMMA | LPAREN | RPAREN | PLUS | MINUS | TIMES | DIV | MOD |
  INT of int | ID of char list | EOF ;;

let isDigit c = c >= '0' && c <= '9' ;;

let isLetter c = c >= 'a' && c <= 'z' ;;

let toDigit c = Char.code c - Char.code '0' ;;

let sep = function (* true if input is empty or starts with a separator *)
  | [] -> true
  | c::_ -> not (isDigit c || isLetter c) ;;

let rec symbol (l, s) =
  match s with
    | c::t when c <= ' ' -> symbol (l + 1, t)
    | ','::t -> (COMMA, (l + 1, t))
    | '('::t -> (LPAREN, (l + 1, t))
    | ')'::t -> (RPAREN, (l + 1, t))
    | '+'::t -> (PLUS, (l + 1, t))
    | '-'::t -> (MINUS, (l + 1, t))
    | '*'::t -> (TIMES, (l + 1, t))
    | 'd'::'i'::'v'::t when sep t -> (DIV, (l + 3, t))
    | 'm'::'o'::'d'::t when sep t -> (MOD, (l + 3, t))
    | c::t when isDigit c ->
        let rec num n = function (* returns integer and remainder of source *)
          | l, d::u when isDigit d -> num (n * 10 + toDigit d) (l + 1, u)
          | any -> (n, any)
        in let (n, u) = num (toDigit c) (l + 1, t) in (INT n, u)
    | c::t when isLetter c ->
        let rec id s = function (* returns char list and remainder of source *)
          | l, d::u when isLetter d || isDigit d -> id (s @ [d]) (l + 1, u)
          | any -> (s, any)
        in let (r, u) = id [c] (l + 1, t) in (ID r, u)
    | _ -> (EOF, (l, s)) ;;

let rec expression s =
  let rec moreterms (p, t) =
    match symbol t with
      | PLUS, u -> let (q, v) = term u in moreterms (Sum (p, q), v)
      | MINUS, u -> let (q, v) = term u in moreterms (Diff (p, q), v)
      | _ -> (p, t)
  in moreterms (term s)

and term s =
  let rec morefactors (p, t) =
    match symbol t with
    | TIMES, u -> let (q, v) = factor u in morefactors (Prod (p, q), v)
    | DIV, u -> let (q, v) = factor u in morefactors (Quot (p, q), v)
    | _ -> (p, t)
  in morefactors (factor s)

and factor s =
  match symbol s with
    | LPAREN, t ->
        let p, u = expression t in
        (match symbol u with
          | RPAREN, v -> (p, v)
          | _, (p, _) -> raise (Source_error (p, "missing )")))
    | INT n, t -> (Int n, t)
    | ID id, t ->
        (match symbol t with
          | LPAREN, u ->
              let p, v = exprList u in
              (match symbol v with
                | RPAREN, w -> (Fun (id, p), w)
                | _, (p, _) -> raise (Source_error (p, "missing )")))
          | _ -> (Id id, t))
    | EOF, (p, _) -> raise (Source_error (p, "unexpected end"))
    | _, (p, _) -> raise (Source_error (p, "id or num expected"))

and exprList s =
  let p, t = expression s in
  match symbol t with
    | COMMA, u -> let (q, n) = exprList u in (p::q, n)
    | _ -> ([p], t)
;;

let parse s =
  try
    match expression (1, s) with
      p, (l, t) ->
        print_string (string_of_int (l - 1) ^ " characters parsed"); p
  with Source_error (p, m) ->
    raise (Failure ("error at " ^ string_of_int p ^ ": " ^ m ^ "")) ;;
```

For example:

```ocaml
# parse ['a'; ' '; '+'; ' '; '('; 'b'; ' '; 'd'; 'i'; 'v'; ' '; 'c'; ')'];;
13 characters parsed- : exp = Sum (Id ['a'], Quot (Id ['b'], Id ['c']))
# parse ['1'; '2'; '*'; '3'; '4'; '5'; '*'; 'a'; 'b'; 'c'];;
10 characters parsed- : exp = Prod (Prod (Int 12, Int 345), Id ['a'; 'b'; 'c'])
# parse ['a'; '+'; '('; 'b'];;
Exception: Failure "error at 5: missing )".
```

The scanner, function `symbol`, can use an arbitrary lookahead; however, the parsing functions can use only a single symbol lookahead, as the `match`-statements are of the form `match symbol s with`. While this form alleviates the need to explicitly construct a list of all symbols, it partly negates the benefit of functional parsing.

### Combinator Parsing

Since parsing functions are just functions that can be passed as parameters, the idea of *combinator parsing* is to construct new parsers by *combinator function*. These are defined to reflect the structure of EBNF grammars. Combinator parsing relies on the use of exceptions:

```ocaml
exception Noparse;;
```

The function `sym x` tests whether the first symbol of the input is `x`; if so, it returns the remainder of the input, otherwise it raises an exception:

```ocaml
let sym x = function
  h::t when x = h -> t | _ -> raise Noparse ;;
```

The function `option pr s` applies parser `pr` to `s` and returns the remainder in that case. If that fails, it returns `s` instead:

```ocaml
let option pr s =
  try pr s
  with Noparse -> s ;;
```

The function `repeat pr s` keeps applying parser `pr` to `s` until parsing fails, and then returns the remainder of the input:

```ocaml
let rec repeat pr s =
  try repeat pr (pr s)
  with Noparse -> s ;;
```

The function `seq pr1 pr2 s` first applies parser `pr1` to `s`, then parser `pr2` to the remainder, and returns the remainder. Either `pr1` or `pr2` may fail:

```ocaml
let seq pr1 pr2 s = pr2 (pr1 s);;
```

The function `choice pr1 pr2 s` applies parser `pr1` to `s`. If that succeeds, the remainder of the input is returned; otherwise, `pr2` is applied to `s`, which may return the remainder or fail.

```ocaml
let choice pr1 pr2 s =
  try pr1 s
  with Noparse -> pr2 s ;;
```

As before, one parsing function is defined for each nonterminal; the function body is constructed by: 

| `E`       | `Pr E`                 |
|:----------|:-----------------------|
| `'a'`     | `sym 'a'`              |
| `B`       | `B`                    |
| `(E₁)`    | `Pr E₁`                |
| `[E₁]`    | `option (Pr E₁)`       |
| `{E₁}`    | `repeat (Pr E₁)`       |
| `E₁ E₂`   | `seq(Pr E₁, Pr E₂)`    |
| `E₁ │ E₂` | `choice(Pr E₁, Pr E₂)` |

For example, the parser for `A = {a} b` is:

```ocaml
let aa = seq (repeat (sym 'a')) (sym 'b');;
```

If the input is only partially recognized, the remainder is returned:

```ocaml
# aa ['a'; 'a'; 'b'];;
- : char list = []
# aa ['a'; 'b'; 'b'];;
- : char list = ['b']
```

The grammar `A = a a A | a b` is not `LL(1)` but is `LL(2)`. The parser is:

```ocaml
let rec aa s =
  choice (seq (sym 'a') (seq (sym 'a') aa)) (seq (sym 'a') (sym 'b')) s;;
```

As combinator parsers backtrack in case of failure, parsing still works:

```ocaml
# aa ['a'; 'a'; 'b'];;
Exception: Noparse.
# aa ['a'; 'a'; 'a'; 'b'];;
- : char list = []
```

The following grammar is not `LL(k)` for any `k`:

    S = A | B
    A = x A | y
    B = x B | z

As backtracking may go back to the beginning of the input, parsing works even for this grammar:

```ocaml
let rec ss s = choice aa bb s
and aa s = choice (seq (sym 'x') aa) (sym 'y') s
and bb s = choice (seq (sym 'x') bb) (sym 'z') s;;
```

However, in general, the effort may be exponential in the length of the input:

```ocaml
# ss ['x'; 'x'];;
Exception: Noparse.
# ss ['x'; 'x'; 'x'; 'x'; 'x'; 'x'; 'z'];;
- : char list = []
```

Left-recursive grammars are not suitable for combinator parsing, as they would lead to infinite recursion.

Combinator parsing libraries exist for all modern programming languages. A difficulty with combinator parsing is producing suitable error messages: without context, the parser `sym 'a'` will fail if the input does not start with `a`, whether `a` must be present or an alternative is possible.

### Earley's Parser

[Earley's parser](https://doi-org.libaccess.lib.mcmaster.ca/10.1145/362007.362035) works with an arbitrary context-free grammar without backtracking. If the grammar is unambiguous, it produces a parse tree in quadratic time; if the grammar is ambiguous, it produces all parse trees in cubic time (in the length of the input). For most “practical” grammars, it produces a parse tree in linear time.

We assume that the start symbol `S` appears only on the left-hand side of one rule, `S → π`; if that is not the case, a rule `S' → S` with a new start symbol `S'` can be added. Earley's parser is a top-down parser that constructs all possible derivations simultaneously: starting with `S`, nonterminals are eagerly expanded according to all possible productions, rather than just a single production.

Let `P` be the set of productions and let the input be given by `x₁, …, xₙ`. Assume `xₙ₊₁ = $`, where `$` is a symbol that does not occur anywhere in the grammar. For each position of the input, a set `sᵢ` of *Earley items* is maintained. An (Earley) item is a grammar rule with the right-hand side split, visualized by `•`, together with an index into the input string. An item `(A → σ • ω, j)`at position `i` means that `A` is attempted to be recognized at input position `j + 1` and up to `i` the input `xⱼ₊₁…xᵢ` can be derived from `σ`, formally `σ ⇒* xⱼ₊₁…xᵢ`. At each position `i`, the algorithm adds items to `sᵢ` in *predict* and *complete* steps and to `sᵢ₊₁` in *match* steps. The algorithm iterates over all items at one position. Since items are being added, a set, `v`, of visited items is maintained.

```
s₀ := {(S → • π, 0)}; for i = 1 to n do sᵢ := {}
for i = 0 to n do
    v := {}
    while v ≠ sᵢ do
        e :∈ sᵢ - v; v := v ∪ {e}
        case e of
            (A → σ • a ω, j) and a = xᵢ₊₁:        -- match (M)
                sᵢ₊₁ := sᵢ₊₁ ∪ {(A → σ a • ω, j)} 
            (A → σ • B ω, j):                            -- predict (P)
                for B → μ ∈ P do
                    sᵢ := sᵢ ∪ {(B → • μ, i)} 
            (A → σ •, j):                                   -- complete (C)
                for (B → μ • A ξ, k) ∈ sⱼ do
                    sᵢ := sᵢ ∪ {(B → μ A • ξ, k)}
accept := (S → π •, 0) ∈ sₙ
```

<div style="float:left">

Consider grammar `G₁`:

```
    E → T | E + T
    T → F | T × F
    F → a
```

<br>

The input `a + a × a` is accepted as `(S → E •, 0) ∈ s₅`.

The underlined lines constitute the derivation.

</div>

<div style="float:right;margin-left:2em">

|            | item                | step      |
|:-----------|:--------------------|:----------|
| `s₃`:      | <u> `F → a •, 2` </u>       | M at `2`  |
| `(x₄ = ×)` | <u> `T → F •, 2` </u>        | C         |
|            | `E → E + T •, 0`    | C         |
|            | `T → T • × F, 2`    | C         |
|            | `S → E •, 0`        | C         |
|            | `E → E • + T, 0`    | C         |
| `s₄`:      | `T → T × • F, 2`    | M at `3`  |
| `(x₅ = a)` | `F → • a, 4`        | P         |
| `s₅`:      | <u> `F → a •, 4` </u>        | M at `4`  |
| `(x₆ = $)` | <u> `T → T × F •, 2` </u>    | C         |
|            | <u> `E → E + T •, 0` </u>    | C         |
|            | `T → T • × F, 2`    | C         |
|            | <u> `S → E •, 0` </u>        | C         |
|            | `E → E • + T, 0`    | C         |

</div>
<div style="float:right">

|            | item                | step      |
|:-----------|:--------------------|:----------|
| `s₀`:      | `S → • E, 0`        |           |
| `(x₁ = a)` | `E → • T, 0`        | P         |
|            | `E → • E + T, 0`    | P         |
|            | `T → • F, 0`        | P         |
|            | `T → • T × F, 0`    | P         | 
|            | `F → • a, 0`        | P         |
| `s₁`:      | <u> `F → a •, 0` </u>       | M at `0`  |
| `(x₂ = +)` | <u> `T → F •, 0` </u>        | C         |
|            | <u> `E → T •, 0` </u>        | C         |
|            | `T → T • × F, 0`    | C         |
|            | `S → E •, 0`        | C         |
|            | `E → E • + T, 0`    | C         |
| `s₂`:      | `E → E + • T, 0`    | M at `1`  |
| `(x₃ = a)` | `T → • T × F, 2`    | P         |
|            | `T → • F, 2`        | P         |
|            | `F → • a, 2`        | P         |

</div>


The Python implementation below assumes that each terminal and nonterminal is a single character, the grammar is represented by a tuple of productions, and each production is a string of the form `A→τ` where `A` is a nonterminal. The first production, `g[0]` in the implementation, defines the start symbol. Since in Python strings are indexed starting from `0`, an extra character, `^`, is prepended to the input. The sequence `a ω` in the algorithm corresponds to `τ` in the implementation, and `A ξ` corresponds to `ν`. 

In [ ]:
def parse(g: tuple, x: str, log = False) -> bool:
    global s
    n = len(x); x = '^' + x + '$'; S, π = g[0][0], g[0][2:]
    s = [{(S, '', π, 0)}] + [set() for _ in range(n)]
    if log: print('   s[0]: ', S, '→ •', π, ', 0', sep='')
    for i in range(n + 1):
        v = set() # visited items
        while v != s[i]:
            e = (s[i] - v).pop(); v.add(e) # pick an arbitrary un-visited item
            A, σ, τ, j = e
            if len(τ) > 0 and τ[0] == x[i + 1]: # match, a == τ[0]
                f = (A, σ + τ[0], τ[1:], j)
                s[i + 1].add(f)
                if log: print('M  s[', i + 1, ']: ', f[0], '→', f[1], '•', f[2], ', ', f[3], sep='')
            elif len(τ) > 0: # predict, B == ω[0]
                for f in ((r[0], '', r[2:], i) for r in g if r[0] == τ[0]):
                    s[i].add(f)
                    if log: print('P  s[', i, ']: ', f[0], '→', f[1], '•', f[2], ', ', f[3], sep='')
            else: # complete, len(τ) == 0
                for f in ((B, μ + ν[0], ν[1:], k) for (B, μ, ν, k) in s[j] if len(ν) > 0 and ν[0] == A):
                    s[i].add(f)
                    if log: print('C  s[', i, ']: ', f[0], '→', f[1], '•', f[2], ', ', f[3], sep='')
    return (S, π, '', 0) in s[n]

In [ ]:
G1 = ("S→E", "E→a", "E→E+E")

In [ ]:
assert parse(G1, "a+a+a")

The steps of the algorithm can be observed by the `log` parameter:

In [ ]:
assert parse(G1, "a+a+a", True)

The set `s` of items is global and can be inspected:

In [ ]:
G2 = ("S→E", "E→T", "E→E+T", "T→F", "T→T×F", "F→a")

In [ ]:
assert parse(G2, "a+a×a"); s

For efficiency, instead of using a set of items for an Earley state, a list with a marker separating visited and unvisited items can be used.

The number of items in `sᵢ` is proportional to `i` in the worst case. For an input of length `n`, in the order of `i²` items are needed in the worst case. Matching and predicting need at most `i` steps for `sᵢ`, but completing may need `i²` steps, as adding an item may cause a previous set to be revisited. Summing `i²` for `i` from `0` to `n` is `n³`, thus Earley's parser needs `n³` steps in the worst case.

### Parsing Expression Grammars and Packrat Parsing

The productions of context-free grammars are *generative*. [*Parsing expression grammars*](https://bford.info/pub/lang/peg/) are an alternative to context-free grammars that describe how practical parsers *recognize:*
- They do not allow nondeterminism in productions; these are resolved in the grammar using prioritized choice and greedy operators.
- *Syntactic predicates* allow to express certain non-context-free languages.
- Parsing allows unlimited lookahead and backtracking, but still runs in time linear to the length of the input.
- Parsers can be implemented by recursive descend and are simple enough to be written by hand.

Productions are written as `A ← E`, and `E` is a *parsing expression*. Parsing according to a parsing expression may *succeed* or *fail:*

| expression     | name               |                           |
|:---------------|:-------------------|:--------------------------|
| `'ε'`          | empty string       | succeed without consuming |
| `'a'`          | literal string     | consume `a` literally, otherwise fail |
| `.`            | any symbol         | consume any symbol, fail at the end of input |
| `B`            | nonterminal `B`    | consume `B`, otherwise fail |
| `(E)`          | grouping           | consume `E`, otherwise fail |
| `E?`           | optional           | consume `E` if possible |
| `E*`           | zero-or-more       | consume `E` as often as possible |
| `E+`           | one-or-more        | consume `E` once, otherwise fail, and then as often as possible |
| `&E`           | and-predicate      | match `E` and do not consume, otherwise fail  |
| `!E`           | not-predicate      | match anything but `E`  and do not consume, otherwise fail |
| `E₁ E₂  …`     | sequence           | consume `E₁`, then `E₂`,  …, otherwise fail |
| `E₁ / E₂ /  …` | prioritized choice | consume `E₁`, otherwise consume `E₂`,  …, otherwise fail |

In EBNF, 

  `A → a b | a`    and    `A → a | a b`

are equivalent. The PEG rules

  `A ← a b / a`    and    `A ← a / a b`

are different: the second rule will never match `a b` as the first alternative is given priority. For example, the EBNF production

    statement → "if" expression "then" statement ["else" statement"]
    
allows `if E then if F then S else T` to be parsed as either `if E then (if F then S) else T` or as `if E then (if F then S else T)`, known as the *dangling else* problem. In EBNF, this is resolved informally or by complicating the grammar. In PEG, a prioritized choice or optional expression resolves the ambiguity:

    statement ← "if" expression "then" statement "else" statement" / "if" expression "then" statement
    statement ← "if" expression "then" statement ("else" statement")?

That is, `E?` is *greedy*: `E` must be consumed if possible; it is a shorthand for `E / ε`.

Consider the definition of symbols in terms of characters:

    operator → '<' ' =' | '<' | '=' | '<' '<'

The sentence `<<=` is an ambiguous sequence of symbols. This is resolved informally by applying the *longest-match rule*. The longest match can be expressed in PEG:

    operator ← '<' ' =' / '<' '<' / '<' / '=' 
    
The regular expression `a* a` matches a non-empty sequence of `a`s and is equivalent to `a a*`. The PEG expression `a* a` will not match any sequence of `a`s as `a*` matches the whole sequence, and the final `a` cannot be matched. Greedy repetition is equivalent to recursion with prioritized choice:

  `A ← E*`    is equivalent to    `A ← E A / ε`  
  `A ← E+`    is equivalent to    `A ← E E*`

For example, with `• ` indicating how much is matched,

  `('0' / '1')*`    matches    `110• +10`

The *and-predicate* and *not-predicate* match but do not consume their operands. For example,

  `'f' 'o' 'o' 'd' &('i' 'e')`    matches    `food•ie`    and fails on    `foodchain`  
  `'f' 'o' 'o' 'd' !('i' 'e')`    fails on     `foodie`    and matches  `food•chain`

A parsing expression grammar `P = (T, N, R, S)` is specified by 
- a finite set `T` of *terminal symbols*,
- a finite set `N` of *nonterminal symbols*,
- a finite set `R` of *rules*,
- a symbol `S ∈ N`, the *start symbol*

where `N ∩ T = {}` and `V = T ∪ N` is its *vocabulary*. Rules are  pairs, written `A ← E`, where `A` is a nonterminal and `E` is a parsing expression. The grammar must not contain direct or indirect left recursion.

The language `L(P)` accepted by `P` is the language accepted by a parser for `P` constructed as follows. A *recursive descent parser with backtracking* consists of a set of mutually recursive parsing procedures reading a global variable `src` with the input. A parsing procedure for nonterminal `A` takes an integer parameter at which parsing the input should start and returns either the index to where `A` was recognized or `Fail`. Parsing procedures must not have any side effects. The call `r ← A(k)` either returns an integer and `A ⇒* src[k:r]` or returns `Fail` as `A` cannot be parsed starting at `k`:

| `p`             | `pr(p)`                      |
|:----------------|:-----------------------------|
| `B ← E`         | `procedure B(k: integer) → integer \| Fail` <br> `pr(E)` <br> `return k` |

The rules for constructing `pr(E)` for recognizing `E` starting at position `k` in `src` are:

| `E`            | `pr(E)`                             |
|:---------------|:------------------------------------|
| `'ε'`          | `skip` |
| `'a'`          | `if prefix('a', src[k:]) then k := k + len(a) else k := Fail` |
| `.`            | `if k < len(src) then k := k + 1 else k := Fail` |
| `B`            | `k ← B(k)`                               |
| `(E₁)`         | `pr(E₁)`                            |
| `E₁?`          | `var r  := k` <br> `pr(E₁)` <br> `if k = Fail then k := r` |
| `E₁*`          | `var r` <br> `while k ≠ Fail do` <br>  `r := k ; pr(E₁)` <br> `k := r` |
| `E₁+`          | `pr(E₁)` <br> `var r := k` <br> `while k ≠ Fail do` <br>  `r := k ; pr(E₁)` <br> `k := r` |
| `&E`           | `var r := k` <br> `pr(E₁)` <br> `if k ≠ Fail then k := r` |
| `!E`           | `var r := k` <br> `pr(E₁)` <br> `if k = Fail then k := r else k := Fail` |
| `E₁ E₂  …`     | `pr(E₁)` <br>`if k ≠ Fail then pr(E₂)` <br>`…`       |
| `E₁ / E₂ /  …` | `var r := k` <br> `pr(E₁)` <br> `if k = Fail then` <br>  `k := r ; pr(E₂)` <br>  `…`  |

The procedure of the start symbol must be called to recognize a sentence in the language.

Consider the following EBNF grammar; it cannot be parsed by recursive descent with `k` symbol lookahead for any `k`:

    S → A | B
    A → x A | y
    B → x B | z

The corresponding PEG, called `P₀`, is:

    S ← A / B
    A ← x A / y
    B ← x B / z

The resulting recursive descent parser with backtracking is expressed as a Python class with the source as a field; failing parsing procedures return `None`:

In [ ]:
class P0Backtrack:
    src: str

    def literal(self, k: int, a: str) -> int | None:
        if self.src.startswith(a, k): return k + len(a) # else return None

    def S(self, k: int) -> int | None: # S ← A / B
        r = k; k = self.A(k)
        if k == None: k = self.B(r)
        return k

    def A(self, k: int) -> int | None: # A ← x A / y
        r = k; k = self.literal(k, 'x')
        if k != None: k = self.A(k)
        if k == None: k = self.literal(r, 'y')
        return k

    def B(self, k: int) -> int | None: # B ← x B / z
        r = k; k = self.literal(k, 'x')
        if k != None: k = self.B(k)
        if k == None: k = self.literal(r, 'z')
        return k

    def parse(self, s: str) -> bool:
        self.src = s
        return self.S(0) == len(s)

In [ ]:
p0 = P0Backtrack()
assert not p0.parse('') and not p0.parse('x') and p0.parse('y') and p0.parse('z')
assert p0.parse('xy') and p0.parse('xz') and not p0.parse('xx') and p0.parse('xxz')

Grammar `P₁` corresponds to grammar `G₁` above: 

    E ← T ('+' T)*  
    T ← F ('×' F)*
    F ← 'a'

Here is its backtracking parser; method `literal(k, a)` is modified to display the matches of terminals:

In [ ]:
class P1Backtrack:
    src: str

    def literal(self, k: int, a: str) -> int | None:
        if self.src.startswith(a, k): print('match', a, 'at', k); return k + len(a)
        else: print('fail', a, 'at', k); return None

    def E(self, k: int) -> int | None: # E ← T ('+' T)*
        k = self.T(k)
        if k != None:
            while k != None:
                r = k 
                k = self.literal(k, '+')
                if k != None: k = self.T(k)
            k = r
        return k

    def T(self, k: int) -> int | None: # T ← F ('×' F)*
        k = self.F(k)
        if k != None:
            while k != None:
                r = k 
                k = self.literal(k, '×')
                if k != None: k = self.F(k)
            k = r
        return k

    def F(self, k: int) -> int | None: # F ← 'a'
        k = self.literal(k, 'a')
        return k
    
    def parse(self, s: str) -> bool:
        self.src = s
        return self.E(0) == len(s)

In [ ]:
p1 = P1Backtrack()
assert p1.parse('a+a×a')

The output shows that matching `×` is attempted before matching `+`, giving `×` higher precedence  that `+`.

The following *parsing attribute grammar* counts the number of pairs of `a` symbols in the input. For example, `aabaaabaaba` has two pairs. A not-predicate is used to express that `aa` must not be followed by another `a` to count as a pair. Recognizing `aa` is given preference to recognizing any other sequence of `a` and `b` inputs.

    S(c) ← « c := 0 » (Pair « c := c + 1 » / Other)*
    Pair ← aa !a
    Other ← a+ / b+

The parsing procedures return a tuple with the index and the synthesized attributes. Here, `S` returns a pair with the index to where the input is recognized and the count of pairs. The attribute rule `« c := c + 1 »` is executed only if `Pair` is successfully parsed.

In [ ]:
class P2Backtrack:
    src: str

    def literal(self, k: int, a: str) -> int | None:
        if self.src.startswith(a, k): return k + len(a) # else return None

    def S(self, k: int) -> tuple[int | None, int]:
        # S(c) ← « c := 0 » (Pair « c := c + 1 » / Other)*
        c = 0
        while k != None:
            r = k 
            k = self.Pair(k)
            if k == None: k = self.Other(r)
            else: c += 1
        return (r, c)

    def Pair(self, k: int) -> int | None:
        # Pair ← aa !a
        k = self.literal(k, 'aa')
        if k != None:
            r = k; k = self.literal(k, 'a')
            k = r if k == None else None
        return k

    def Other(self, k: int) -> int | None:
        # Other ← a+ / b+
        r = k
        k = self.literal(r, 'a')
        if k != None:
            while k != None:
                r = k; k = self.literal(k, 'a')
            k = r
        if k == None:
            k = self.literal(r, 'b')
            if k != None:
                while k != None:
                    r = k; k = self.literal(k, 'b')
                k = r
        return k
    
    def parse(self, s: str) -> tuple[int | None, int]:
        self.src = s; return self.S(0)

In [ ]:
p2 = P2Backtrack()
assert p2.parse('aaaa') == (4, 0) and p2.parse('aabaaabaaba') == (11, 2) and p2.parse('c') == (0, 0)

Let us examine the effects of backtracking with grammar `P₃` given by:

  `S ← Y / Z` (1)   `Y ← x X y` (2)   `Z ← x X z` (3)   `X ← S?` (4)

The table below shows the steps for parsing `x x z z`. The `•` indicates how much of the input is matched:

<div style="float:left;margin-right:2em">

| Step | Rule |
|:------|:------|
|` • S` |  |
|`⇒  • Y`| (1) |
| <code>⇒  x • <u>X</u> y</code> | (2) |
| <code>⇒  x • <u>S</u> y</code> | (4) |
| <code>⇒  x • <u>Y</u> y</code> | (1) |
| <code>⇒  x <u>x • X y</u> y</code> | (2) |
| <code>⇒  x <u>x • S y</u> y</code> | (4) |
| <code>⇒  x <u>x • Y y</u> y</code> | (1) |
| <code>⇒  x <u>x • x X y y</u> y</code> | (2) |
| <code>⇒  x <u>x • Y y</u> y</code> | backtracking (2) |
| <code>⇒  x <u>x • S y</u> y</code> | backtracking (1) |
| <code>⇒  x <u>x • X y</u> y</code> | backtracking (4) |
| <code>⇒  x <u>x • y</u> y</code> |  (4) |
| <code>⇒  x <u>x • X y</u> y</code> | backtracking (4) |
| <code>⇒  x • <u>Y</u> y</code> | backtracking (2) |
| <code>⇒  x • <u>S</u> y</code> | backtracking (1) |
| <code>⇒  x • <u>Z</u > y</code> |  (1) |
| <code>⇒  x <u>x • X z</u> y</code> |  (3) |
| <code>⇒  x <u>x • S z</u> y</code> |  (4) |
| <code>⇒  x <u>x • Y z</u> y</code> |  (1) |
| <code>⇒  x <u>x • x X y z</u> y</code> |  (2) |
| <code>⇒  x <u>x • Y z</u> y</code> | backtracking (2) |

</div>

<div style="float:left;margin-right:2em">

| Step | Rule |
|:------|:------|
| <code>⇒  x <u>x • S z</u> y</code> | backtracking (1) |
| <code>⇒  x <u>x • X z</u> y</code> | backtracking (4) |
| <code>⇒  x <u>x • z</u> y</code> |  (4) |
|`⇒  x x • X z y` | backtracking (4) |
|`⇒  x • Z y` | backtracking (3) |
|`⇒  x • S y` | backtracking (1) |
|`⇒  x • X y` | backtracking (4) |
|`⇒  x • y` |  (4) |
|`⇒  x • X y` | backtracking (4) |
|`⇒  • Y` | backtracking (2) |
|`⇒  • S` | backtracking (1) |
|`⇒  • Z` |  (1) |
| <code>⇒  x • <u>X</u> z</code> |  (3) |
| <code>⇒  x • <u>S</u> z</code> |  (4) |
| <code>⇒  x • <u>Y</u> z</code> |  (1) |
| <code>⇒  x <u>x • X y</u> z</code> |  (2) |
| <code>⇒  x <u>x • S y</u> z</code> |  (4) |
| <code>⇒  x <u>x • Y y</u> z</code> |  (1) |
| <code>⇒  x <u>x • x X y y</u> z</code> |  (2) |
| <code>⇒  x <u>x • Y y</u> z</code> | backtracking (2) |
| <code>⇒  x <u>x • S y</u> z</code> | backtracking (1) |
| <code>⇒  x <u>x • Z y</u> z</code> |  (1) |

</div>

<div style="float:left">

| Step | Rule |
|:------|:------|
| <code>⇒  x <u>x • x X z y</u> z</code> |  (3) |
| <code>⇒  x <u>x • Z y</u> z</code> | backtracking (3) |
| <code>⇒  x <u>x • S y</u> z</code> | backtracking (1) |
| <code>⇒  x <u>x • X y</u> z</code> | backtracking (4) |
| <code>⇒  x <u>x • y</u> z</code> |  (4) |
| <code>⇒  x <u>x • X y</u> z</code> | backtracking (4) |
|<code>⇒  x • <u>Y</u> z</code> | backtracking (2) |
|<code>⇒  x • <u>S</u> z</code> | backtracking (1) |
|<code>⇒  x • <u>Z</u> z</code> |  (1) |
| <code>⇒  x <u>x • X z</u> z</code> |  (3) |
| <code>⇒  x <u>x • S z</u> z</code> |  (4) |
| <code>⇒  x <u>x • Y z</u> z</code> |  (1) |
| <code>⇒  x <u>x • x X y z</u> z</code> |  (2) |
| <code>⇒  x <u>x • Y z</u> z</code> | backtracking (2) |
| <code>⇒  x <u>x • S z</u> z</code> | backtracking (1) |
| <code>⇒  x <u>x • Z z</u> z</code> |  (1) |
| <code>⇒  x <u>x • Z z</u> z</code> |  (1) |
| <code>⇒  x <u>x • x X z z</u> z</code> |  (3) |
| <code>⇒  x <u>x • Z z</u> z</code> | backtracking (3) |
| <code>⇒  x <u>x • S z</u> z</code> | backtracking (1) |
| <code>⇒  x <u>x • X z</u> z</code> | backtracking (4) |
| <code>⇒  x <u>x z</u> z •</cide> |  (4) |

</div>

The underlined symbols highlight two series of steps for `X ⇒* x z`, starting at input position `1`. The first fails as `z` does not match the input at position `2`. The second series succeeds. 

The parser with backtracking for this grammar is:

In [ ]:
class P3Backtrack:
    src: str

    def literal(self, k: int, a: str) -> int | None:
        if self.src.startswith(a, k): return k + len(a) # else return None

    def S(self, k: int) -> int | None: # S ← Y / Z
        r = k; k = self.Y(k)
        if k == None: k = self.Z(r)
        return k

    def Y(self, k: int) -> int | None: # Y ← x X y
        k = self.literal(k, 'x')
        if k != None:
            k = self.X(k)
            if k != None: k = self.literal(k, 'y')
        return k

    def Z(self, k: int) -> int | None: # Z ← x X z
        k = self.literal(k, 'x')
        if k != None:
            k = self.X(k)
            if k != None: k = self.literal(k, 'z')
        return k

    def X(self, k: int) -> int | None: # X ← S?
        r = k; k = self.S(k)
        if k == None: k = r
        return k

    def parse(self, s: str) -> int | None:
        self.src = s; return self.S(0)

In [ ]:
p3b = P3Backtrack()
assert p3b.parse('xxz') == None and p3b.parse('xxzz') == 4

A *packrat parser* avoids backtracking by *memoizing* the result of parsing each nonterminal at each input position. For nonterminal `A`, a map (Python dictionary), say `am`, has an entry at index `k` with the result of parsing `A` at that position, `m[k] = A(k)`: is either the index in the input up to which `A` is parsed or `Fail` (`None` in Python) as `A` cannot be parsed starting at `k`. 

Memoization can be added to a recursive descent parser with backtracking by inheritance:

In [ ]:
class P3Memoizing(P3Backtrack):
    sm: dict[int, int | None] # for memoizing S
    ym: dict[int, int | None] # for memoizing A
    zm: dict[int, int | None] # for memoizing Z
    xm: dict[int, int | None] # for memoizing X
    def S(self, k: int) -> int | None:
        if k not in self.sm: self.sm[k] = super().S(k)
        return self.sm[k]
    def Y(self, k: int) -> int | None:
        if k not in self.ym: self.ym[k] = super().Y(k)
        return self.ym[k]
    def Z(self, k: int) -> int | None:
        if k not in self.zm: self.zm[k] = super().Z(k)
        return self.zm[k]
    def X(self, k: int) -> int | None:
        if k not in self.xm: self.xm[k] = super().X(k)
        return self.xm[k]
    def parse(self, s: str) -> int | None:
        self.sm, self.xm, self.ym, self.zm = {}, {}, {}, {}
        return super().parse(s)

In [ ]:
p3m = P3Memoizing()
assert p3m.parse('xxzz') == 4

Let us inspect the maps constructed for memoizing:

In [ ]:
assert p3m.sm == {0: 4, 1: 3, 2: None}
assert p3m.xm == {1: 3, 2: 2}
assert p3m.ym == {0: None, 1: None, 2: None}
assert p3m.zm == {0: 4, 1: 3, 2: None}

The map for `X` has the value `3` for index `1`, meaning `X ⇒* src[1:3]`. The map has a value `2` for index `2`, meaning `X ⇒* src[2:2]`, which states that `X` was parsed, but no input was consumed. Indeed, within the first series of underlined symbols, there are two attempts to parse `X` at position `2`. The entries at index `0`, `1`, and `2` in the map for `Y` state that parsing `Y` at those indices failed. When parsing `xxzz`, the maps for `S`, `Y`, and `Z` are built but not used. 

Here is a comparison of the runtime of the backtracking and memoizing parsers for `P₃`:

In [ ]:
s = 'x'*20 + 'z'*20
%time p3b.parse(s)

In [ ]:
%time p3m.parse(s)

Concerning the run time and memory consumption of packrat parsers, the key is the effect of memoization: 
- An entry may be needed for each nonterminal and each position of the input.
- Backtracking will never attempt to parse the same nonterminal at the same position of the input. Thus, the run time is bound by the number of nonterminals times the length of the input.

Thus, despite backtracking, packrat parsers run in time linear to the length of the input.

The languages parsed by PEGs are closed under union, intersection, and complement. Consider `P₀ = (T, N₀, R₀, S₀)` and `P₁ = (T, N₁, R₁, S₁)` and construct a new grammar, `P`, with the union of the nonterminals, the union of the rules, and a new start symbol, `S`.

- If `S ← S₀ / S₁` is added to the rules, then `L(P) = L(P₀) ∪ L(P₁)`.
- If `S ← &S₀ S₁` is added to the rules, then `L(P) = L(P₀) ∩ L(P₁)`.
- If `S ← !S₀` is added to the rules, then `L(P) = V* ∖ L(P₀)`.

Some context-sensitive languages that are not context-free can be parsed. For example, the language `aⁿbⁿcⁿ` is parsed by `P₄`:
    
    S ← &(AB !b) a* BC
    AB ← a AB b / ε
    BC ← b BC c / ε

**Question.** How does `P₄` work? What about the following variation?

    S ← &(AB !b) a* &(BC) b*c*
    AB ← a AB b / ε
    BC ← b BC c / ε

Here is a comparison of top-down parsing with `k` symbol lookahead (`LL(k)`), packrat parsing, and Earley's parsing:

|                             | `LL(k)` | packrat | Earley  |
|:----------------------------|:--------|:--------|:--------|
| grammar restrictions        | `LL(k)` conditions, including no left-recursion | no left-recursion | arbitrary recursion |
| language parsed             | subset of context-free languages | some non-context-free languages | exactly all context-free languages |
| nondeterministic constructs | nondeterministic choice and repetition in grammar, greedy implementation in parser | greedy choice and repetition | nondeterministic choice and repetition |
| parser construction¹        | hand-written or generated recursive descend, combinator parsing | hand-written recursive descend or combinator parsing | table-driven |
| memory consumption²         | linear in the depth of nesting in the input | linear in the number of nonterminals times the length of the input  | quadratic in the length of the input |
| run time | linear in the length of the input | linear in length of input | cubic in length of input |
| parser state³               | may be stateful | must be stateless | must be stateless |
| typical application⁴        | parsing large inputs with separate scanning | small inputs, combined scanning and parsing | small inputs, nondeterministic languages |

Notes:
1. A table-driven parser is a generic parser that takes the grammar and the source as input. While `LL(k)` parsers can be table-driven, they would negate the efficiency; `LR(k)` parsers are always table-driven. They have weaker conditions on the grammar; in particular, they allow left-recursion.
2. Functional (top-down) parsing also has the `LL(k)` restriction, but makes it easier to express parsers for `k > 1`. Functional parsers are stateless.
3. Functional combinator parsing does not have the `LL(k)` restriction, but left-recursion is not allowed. The run time may be exponential in the length of the input. Combinator parsers are stateless.
4. The constant factors can be significant for combinator parsing, packrat, and Earley.
5. `LL(k)` and `LR(k)` parsing can have side effects, i.e., updating the scanner state, issuing error messages, and evaluating attribute rules for updating the symbol table or generating code. With packrat, parsing procedures must be functions that return the result of attribute computations, since they may need to be discarded during backtracking. With Earley, the result of attribute computations must be attached to the Earley items, and not all items will be needed.
6. The greedy choice and repetition make parsing expression grammars suitable for *scannerless parsing*.  In *domain-specific languages* like database query languages, spreadsheet formulae, URLs, configuration files, search-and-replace expressions, and symbolic math systems, the time for parsing may be negligible compared to processing the input, and the languages are "small" enough that separating scanning from parsing is not worth the additional complexity.

### Probabilistic Grammars

A probabilistic grammar attaches probabilities to productions. With a probabilistic grammar, 

- given a sentence, the probability of that sentence can be determined, and
- sentences can be generated according to the specified probabilities.

A *probabilistic context-free grammar* is a tuple `G = (T, N, P, S, p)` where `(T, N, P, S)` is a context-free grammar and `p` is a function from `P` to `[0..1]` specifying the probability of each production. The probabilities of the productions for each nonterminal must sum up to `1`, i.e., if the productions of `A ∈ N` are `A → τ₀ | τ₁ | …`,

    p(A → τ₀) + p(A → τ₁) + … = 1

The probability of a sequence derived from `G` is the product of all productions of the leftmost derivation.

**Example.** Consider `G₁ = (T, N, P, S, p)`, where `T = {a, b, c, d}`, `N = {S, X}`, `P = {S → a X, S → b X, X → c, X → d}`, `p(S → a X) = 0.4`, `p(S → b X) = 0.6`, `p(X → c) = 0.2`, `p(X → d) = 0.8`. Sentence `a c` is derivable from `S`,

    S ⇒ a X ⇒ a c

and its probability is:

    p(a c) = p(S → a X) × p(X → c) = 0.4 × 0.2 = 0.08

Likewise, 

    p(a d) = p(S → a X) × p(X → d) = 0.4 × 0.8 = 0.32
    p(b c) = p(S → b X) × p(X → c) = 0.6 × 0.2 = 0.12
    p(b d) = p(S → b X) × p(X → d) = 0.6 × 0.8 = 0.48

The probabilities of all sentences add up to `1`.

This model of probabilities has the following properties:

<dl>
  <dt>Place invariance</dt>
  <dd>The probability of a subsequence does not depend on where in a sequence it occurs.</dd>

  <dt>Context-free</dt>
  <dd>The probability of a subsequence does not depend on surrounding words.</dd>

  <dt>Ancestor-free</dt>
  <dd>The probability of a sequence does not depend on ancestors outside its derivation.</dd>
</dl>


For writing probabilistic grammars, we use `@` to specify the probabilities. Grammar `G₁` is written as:

    S → a X @ 0.4
    S → b X @ 0.6
    X → c @ 0.2
    X → d @ 0.8

Probabilistic grammars can be used for natural languages with ambiguous grammars to disambiguate sentences: given a sentence with two parse trees, the probabilities of the two corresponding derivations are used to pick the more likely interpretation of a sentence.

Here, we use probabilistic grammars to generate sentences. Procedure `L()` for generating all sentences of a language is extended to return the probability of a sentence together with the sentence:
    
    procedure G.L(): G.T* 
        dd, d, p := ∅, {G.S}, {S ↦ 1}
        while d ≠ ∅ do
            dd, d := dd ∪ d, ∅
            for μ σ ν ∈ dd, σ → τ @ q ∈ G.P if μ ∈ G.T* do
                χ := μ τ ν
                if χ ∉ dd and χ ∉ d then
                    d := d ∪ {χ}; p(χ) = p(μ σ ν) × q
                    if χ ∈ G.T* then yield χ, p(χ)

The Python implementation uses strings for the probabilities in the grammar and decimal arithmetic for the calculations to avoid rounding errors of floating-point representations.

In [ ]:
from collections.abc import Iterator
from decimal import Decimal

class ProbabilisticGrammar:
    def __init__(self, T: set[str], N: set[str], P: set[tuple[str, str, str]], S: str):
        self.T, self.N, self.P, self.p, self.S = T, N, {A: [] for A in N}, {A: [] for A in N}, S
        for A, τ, q in P: self.P[A].append(τ); self.p[A].append(Decimal(q))

    def L(self, log = False, stats = False) -> Iterator[str]:
        dd, d, p = [], [self.S], {self.S: Decimal(1)}
        if log: print('    ', self.S, '@', 1)
        while d != []:
            if stats: print('# added derivations:', len(d))
            if log: print()
            dd.extend(d); d = []
            for π in dd:
                A = next((x for x in π.split() if x in self.N), None) # find the first nonterminal
                if A != None:
                    for τ, q in zip(self.P[A], self.p[A]): # productions A → τ @ q
                        χ = π.replace(A, τ, 1).replace('  ', ' ').strip()
                        if (χ not in dd) and (χ not in d):
                            d.append(χ)
                            p[χ] = p[χ] + p[π] * q if χ in p else p[π] * q
                            if all(a in self.T for a in χ.split()): yield χ, str(p[χ])
                            if log: print('    ', π, '⇒ᴸ', χ, '@', p[χ])

Here is the above grammar:

In [ ]:
G1 = ProbabilisticGrammar({'a', 'b', 'c', 'd'}, {'S', 'X'},
             {('S', 'a X', '0.4'), ('S', 'b X', '0.6'), ('X', 'c', '0.2'), ('X', 'd', '0.8')},
             'S')

In [ ]:
list(G1.L())

Setting the `log` parameter shows how the derivation with the probabilities:

In [ ]:
list(G1.L(log = True))

Consider the following expression grammar with probabilities:

    E → T @ 0.1
    E → E + T @ 0.9
    T → F @ 0.2
    T → T × F @ 0.8
    F → x @ 0.3
    F → y @ 0.3
    F → z @ 0.3
    F → ( E ) @ 0.1

In [ ]:
G3 = ProbabilisticGrammar({'x', 'y', 'z', '+', '×', '(', ')'}, {'E', 'T', 'F'},
             {('E', 'T', '0.7'), ('E', 'E + T', '0.3'),
              ('T', 'F', '0.8'), ('T', 'T × F', '0.2'),
              ('F', 'x', '0.3'), ('F', 'y', '0.3'), ('F', 'z', '0.3'), ('F', '( E )', '0.1')},
             'E')

The first `20` derived sentences are:

In [ ]:
[d for d, _ in zip(G3.L(), range(20))]

As with context-free grammars, the language can be infinite. Suppose we want to generate a sentence randomly. Starting from the start symbol, a production for that symbol is chosen at random according to the specified probabilities. This process is continued with the leftmost nonterminal of the derived sequence until all symbols are terminals:

In [ ]:
from random import choices

def generate(G: ProbabilisticGrammar, log = False) -> str: # G must be context-free
        χ = G.S
        if log: print('    ', χ, '@', 1)
        while not all(a in G.T for a in χ.split()):
            π = χ
            A = next(x for x in π.split() if x in G.N) # A is the first nonterminal
            i = choices(range(len(G.P[A])), map(float, G.p[A]))[0] # choose production A → G.P[A][i] randomly
            χ = π.replace(A, G.P[A][i], 1).replace('  ', ' ').strip()
            if log: print('    ', π, '⇒ᴸ', χ, '@', G.p[A][i])
        return χ

setattr(ProbabilisticGrammar, 'generate', generate)

In [ ]:
G3.generate(True)

For example, `20` random sentences in `G3` are generated by:

In [ ]:
[G3.generate() for _ in range(20)]

[Fuzzing](https://www.fuzzingbook.org/) is a technique for finding defects in software by randomly generating inputs. Probabilistic grammars can be used to generate test cases when the input adheres to a grammar.